In [ ]:
#Import Pandas library for handling tabular data (DataFrames)
import pandas as pd
# Import NumPy library for numerical operations and arrays
import numpy as np
# Function to split dataset into training and testing sets
from sklearn.model_selection import train_test_split
# LabelEncoder: convert categorical labels to numbers; StandardScaler: normalize features
from sklearn.preprocessing import LabelEncoder, StandardScaler
# Random Forest Classifier: ensemble model for classification tasks
from sklearn.ensemble import RandomForestClassifier
# accuracy_score: measure correct predictions; classification_report: detailed metrics (precision, recall, F1-score)
from sklearn.metrics import accuracy_score, classification_report
# Import Pickle module for saving and loading Python objects (e.g., trained ML models)
import pickle

In [ ]:
df = pd.read_csv("customer_churn.csv")   # Load dataset

df.head()    # Display basic information

In [ ]:
# Returns the list of all column names in the DataFrame
df.columns

In [ ]:
# Gives a summary of the DataFrame: number of rows, columns, data types, non-null counts, and memory usage
df.info()

# Counts missing values in each column: shows how many NaN (null) entries exist per column
df.isnull().sum() 

In [ ]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df.columns   # Returns the list of all column names in the DataFrame

In [ ]:
# Fill missing values
df.fillna(df.median(numeric_only=True), inplace=True)
df.columns   # Returns the list of all column names in the DataFrame

In [ ]:
# Drop unnecessary column
df.drop("customerID", axis=1, inplace=True)
df.columns  # Returns the list of all column names in the DataFrame

In [ ]:
# Converts categorical columns into dummy/indicator variables (one-hot encoding); drop_first=True avoids dummy variable trap by removing the first category
df = pd.get_dummies(df, drop_first=True)   

df.columns   # Returns the list of all column names in the DataFrame

In [ ]:
# Creates feature set X by dropping the target column "Churn_Yes" from the DataFrame; axis=1 means drop a column
X = df.drop("Churn_Yes", axis=1)
# Defines target variable y as the "Churn_Yes" column from the DataFrame
y = df["Churn_Yes"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(   # Splits the dataset into training and testing sets
    X, y, test_size=0.2, random_state=42               # X = features, y = target; test_size=0.2 means 20% data for testing, 80% for training; random_state=42 ensures reproducibility
)


In [ ]:
# Create a StandardScaler object to normalize features (mean=0, variance=1)
scaler = StandardScaler()
# Fit the scaler on training data (learn mean & std) and transform it to scaled values
X_train = scaler.fit_transform(X_train)
# Apply the same scaling (using training mean & std) to the test data
X_test = scaler.transform(X_test)

In [ ]:
# Import the RandomForestClassifier model from scikit-learn
from sklearn.ensemble import RandomForestClassifier
# Create Random Forest model
model = RandomForestClassifier(            # Initialize the model with chosen hyperparameters
    n_estimators=500,                      # Number of trees in the forest (more trees = more robust, but slower)
    max_depth=12,                          # Maximum depth of each tree (controls complexity and prevents overfitting)
    min_samples_split=5,                   # Minimum samples required to split a node (controls tree growth)
    random_state=42                        # Random seed for reproducibility (ensures same results each run)
)

# Train model
model.fit(X_train, y_train)                # Fit the model on training data (learn patterns from features X_train and labels y_train)


In [ ]:
# Use the trained model to predict labels for the test set features (X_test)
y_pred = model.predict(X_test)
     # Calculate accuracy by comparing predicted labels (y_pred) with true labels (y_test)
accuracy = accuracy_score(y_test, y_pred)
# Print the accuracy score (percentage of correct predictions)
print("Accuracy:", accuracy)
# Print a detailed report: precision, recall, F1-score, and support for each class
print(classification_report(y_test, y_pred))


In [ ]:
# Save model files
pickle.dump(model, open("churn_model.pkl","wb"))       # Save the trained Random Forest model into a file named "churn_model.pkl" in write-binary mode
pickle.dump(scaler, open("churn_scaler.pkl","wb"))     # Save the fitted StandardScaler object into a file named "churn_scaler.pkl" (so you can reuse the same scaling later)
pickle.dump(X.columns, open("model_columns.pkl","wb")) # Save the list of feature column names into a file named "model_columns.pkl" (helps ensure correct input format when reloading the model)
